## Points to improve
- ~smoothen slope using 5th and 95th percentiles~
- ~convert decibels to linear~
- ~test code in Kashmir (valley)~
- push low level code to utils and other relevant modules
- final checks on documentation

In [1]:
import sys
sys.path.append('../')

from ifmiap import flood_mapper
from ifmiap import utils
import geopandas as gpd

/opt/homebrew/lib/python3.9/site-packages/geopandas/_compat.py:124: UserWarning: The Shapely GEOS version (3.11.1-CAPI-1.17.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


In [2]:
# small piece of code to get zone-wise list of IDs
gdf = gpd.read_file(r'../resources/india_utm_fishnet.gpkg')
zone_id_group = gdf[['zone', 'ID']].groupby('zone')['ID'].apply(list)

zone_id_dict = dict()

for idx in zone_id_group.index:
    zone_id_dict[idx] = zone_id_group[idx]

In [3]:
# create the flood mapper class
bihar_flood_mapper = flood_mapper(
    grid_shapefile = r'../resources/india_utm_fishnet.gpkg',
    grid_id_list = [148],#zone_id_dict['45R'],#[378, 384, 390],
    dry_date_col = 'dry_month',
    id_col = 'ID',
    dry_years=[2018, 2022],
    slope_dir = r'../resources/slope/',
    wet_duration = ['2022/07', '2022/07']
)

Following previously processed AOI IDs found. Will be skipped. If you are running with a new year range, consider modifying or deleting the json file.
 [148]


In [4]:
%%time
bihar_flood_mapper.get_dry_dates()

if len(bihar_flood_mapper.aoi_ids_to_process) > 0:
    bihar_flood_mapper.generate_dry_date_ranges()
    bihar_flood_mapper.get_s1_items(dry_wet='dry')
    bihar_flood_mapper.read_scenes(dry_wet='dry', overview_level=3)
    bihar_flood_mapper.generate_mean_std_by_aoi()
else:
    bihar_flood_mapper.load_mean_std_by_aoi()
    
bihar_flood_mapper.prepare_slope(dem_overview=0, buffer=500)
bihar_flood_mapper.prepare_wet_scenes(overview_level=3)
bihar_flood_mapper.generate_number_of_scenes(export_raster=True)
bihar_flood_mapper.map_floods(vv_thd=3, vh_thd=3, rel_slope_thd=20,
                              export_raster=False, export_vector=True, export_maps=False)
bihar_flood_mapper.merge_floods_by_date(export_raster=True)
bihar_flood_mapper.monthly_sum()

Previously processed ../output/mean_std/2018_2022_aoi_148_vv_vh_mean_std.nc read successfully!
Slope for tile ID 148 found, will not be downloaded.


/opt/homebrew/lib/python3.9/site-packages/rioxarray/raster_writer.py:130: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.4028234663852886e+38) to match the dtype of the data.
  warnings.warn(
/opt/homebrew/lib/python3.9/site-packages/rioxarray/raster_writer.py:130: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.4028234663852886e+38) to match the dtype of the data.
  warnings.warn(
/opt/homebrew/lib/python3.9/site-packages/rioxarray/raster_writer.py:130: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.4028234663852886e+38) to match the dtype of the data.
  warnings.warn(
/opt/homebrew/lib/python3.9/site-packages/rioxarray/raster_writer.py:130: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.4028234663852886e+38) to match the dtype of the data.
  warnings.warn(
/opt/homebrew/lib/python3.9/site-packages/rioxarray/raster_writer.py:130: UserWa

Flood cells not found in 148_S1A_IW_GRDH_1SDV_20220721T125516_20220721T125545_044199_054682_rtc.
Flood cells not found in 148_S1A_IW_GRDH_1SDV_20220712T004443_20220712T004508_044060_05425C_rtc.
Flood cells not found in 148_S1A_IW_GRDH_1SDV_20220709T125515_20220709T125544_044024_054146_rtc.
CPU times: user 14.9 s, sys: 1.41 s, total: 16.3 s
Wall time: 1min 50s


In [5]:
#bihar_flood_mapper.flush_output(remove_slope=False)